## Ecomotive DLT: Automatización Robusta con Delta Live Tables

![DLT_pipeline.jpg](./DLT_pipeline.jpg "DLT_pipeline.jpg")

Delta Live Tables (DLT) es Declarativo:

- Tú dices QUÉ quieres: "Quiero una tabla Silver que venga de esta tabla Bronze".

- Databricks gestiona el CÓMO: Crea los checkpoints, gestiona las dependencias, reinicia en caso de fallo y escala automáticamente.

Ya no hay por tanto:
- writeStream
- trigger
- checkpointLocation
- gestionar mkdir
- Ddecirle a Python "ejecuta esto AHORA".

##### Sintaxis

![DLT_RUTA.jpg](./DLT_RUTA.jpg "DLT_RUTA.jpg")

##### Expectations (Calidad del Dato)
DLT introduce una forma nativa de gestionar "Datos Sucios" (como nuestro problema de baterías 'CRITICAL_LOW') sin que el pipeline falle silenciosamente. Se llaman Expectations:

- @dlt.expect("desc", "condicion"): Si falla, registra el error en el log pero deja pasar el dato.

- @dlt.expect_or_drop("desc", "condicion"): Si falla, elimina la fila (ideal para Silver).

- @dlt.expect_or_fail("desc", "condicion"): Si falla, detiene todo el pipeline (ideal para errores críticos).

Hacemos click en "Configuration" y definimos la variable que usamos en el código.

Key: pipeline.base_volume_path

Value: /Volumes/ecomotive/landing/ecomotive_vol

In [0]:
import dlt
from pyspark.sql.functions import col, trim, initcap, regexp_replace, to_date, split, current_timestamp, to_timestamp, expr, window, avg, max, count

# --- CONFIGURACIÓN ---
# Leemos la ruta del volumen desde la configuración del Pipeline (lo configuraremos en el paso 3)
base_path = spark.conf.get("bundle.sourcePath")

# -------------------------------------------------------------------------
# BRONZE LAYER (Ingesta con Auto Loader)
# -------------------------------------------------------------------------

@dlt.table(
    comment="Datos crudos de conductores ingestados vía Auto Loader (CSV)",
    table_properties={"quality": "bronze"}
)
def bronze_drivers():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        # Usamos la ruta base configurada
        .load(f"{base_path}/inbox_drivers")
    )

@dlt.table(
    comment="Datos crudos de sensores ingestados vía Auto Loader (JSON)",
    table_properties={"quality": "bronze"}
)
def bronze_sensors():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(f"{base_path}/inbox_sensors")
    )

# -------------------------------------------------------------------------
# SILVER LAYER (Limpieza y Calidad)
# -------------------------------------------------------------------------

@dlt.table(
    comment="Conductores limpios y enriquecidos",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_driver_id", "driver_id IS NOT NULL") # Eliminamos filas sin ID
def silver_drivers():
    return (
        dlt.read("bronze_drivers")
        .withColumn("driver_name", initcap(trim(col("raw_name"))))
        .withColumn("license_ref", trim(col("license_ref")))
        .withColumn("hourly_wage", regexp_replace(col("hourly_wage_text"), " EUR", "").cast("double"))
        .withColumn("hire_date", to_date(col("hired_date"), "yyyy-MM-dd"))
        .withColumn("certifications", split(col("certifications_list"), "-"))
        .withColumn("processed_at", current_timestamp())
        .select(
            "driver_id", "driver_name", "license_ref", 
            "hourly_wage", "certifications", "hire_date", "processed_at"
        )
    )

@dlt.table(
    comment="Sensores aplanados y tipados",
    table_properties={"quality": "silver"}
)
@dlt.expect("valid_rpm", "rpm >= 0") # MEJORA: Solo advertencia si RPM es negativo, no borra datos
def silver_sensors():
    return (
        dlt.read("bronze_sensors")
        .withColumn("rpm", col("engine_data.rpm"))
        .withColumn("temperature", col("engine_data.temperature_c"))
        .withColumn("engine_status", col("engine_data.status"))
        .withColumn("event_time", to_timestamp(col("timestamp")))
        .select(
            "event_id", "sensor_id", "driver_id", "event_time",
            "rpm", "temperature", "engine_status", "battery_level",
            col("location.lat").alias("latitude"),
            col("location.lon").alias("longitude"),
            "error_codes"
        )
    )

# -------------------------------------------------------------------------
# GOLD LAYER (Modelado de Negocio y Agregaciones)
# -------------------------------------------------------------------------

@dlt.table(
    comment="Dimensión de conductores (SCD Tipo 1 implícito)",
    table_properties={"quality": "gold"}
)
def dim_drivers():
    return (
        dlt.read("silver_drivers").select(
            "driver_id", "driver_name", "license_ref", 
            expr("try_cast(hourly_wage as double) as hourly_wage"), 
            "certifications", "hire_date"
        )
    )

@dlt.table(
    comment="Tabla de hechos de sensores",
    table_properties={"quality": "gold"}
)
def fact_sensors():
    return (
        dlt.read("silver_sensors").select(
            "event_id", "sensor_id", "driver_id", "event_time",
            "rpm", "temperature", 
            expr("try_cast(battery_level as double) as battery_level"),
            "engine_status", "latitude", "longitude"
        )
    )

@dlt.table(
    comment="KPIs diarios agregados por conductor",
    table_properties={"quality": "gold"}
)
def daily_driver_kpis():
    # Stream (Fact)
    df_fact = dlt.read("fact_sensors")
    # Static (Dim) - En DLT, leer otra tabla dlt actúa como lectura estática si no es stream puro
    df_dim = dlt.read("dim_drivers")

    return (
        df_fact.join(df_dim, on="driver_id", how="inner")
        .groupBy(
            window(col("event_time"), "1 day").alias("time_window"),
            col("driver_name")
        )
        .agg(
            avg("rpm").alias("avg_rpm"),
            max("temperature").alias("max_temp"),
            avg("battery_level").alias("avg_battery"),
            count("event_id").alias("total_events")
        )
        .select(
            col("time_window.start").alias("date"),
            "driver_name", "avg_rpm", "max_temp", "avg_battery", "total_events"
        )
    )